# Transfer learning on CIFAR-10: VGG16 vs VGG19 vs ResNet50
Frozen ImageNet base + new classifier head, trained on a 5,000-image CIFAR-10 subset.

In Colab: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

In [ ]:
import tensorflow as tf
# take GPU memory only as needed (avoids out-of-memory when several notebooks are open)
for g in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(g, True)
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16, VGG19, ResNet50, vgg16, vgg19, resnet50
from tensorflow.keras.utils import to_categorical

## Load data
Images and labels are sliced **together** so they keep the same length.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
X_train, y_train = X_train[:5000], y_train[:5000]
X_test,  y_test  = X_test[:1000],  y_test[:1000]

y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test, 10)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

## Train each model
Each network uses its own `preprocess_input` (the preprocessing its ImageNet weights were trained with) instead of `/255`.
`clear_session()` frees the previous model's memory before building the next.

In [ ]:
models = [(VGG16, vgg16.preprocess_input),
          (VGG19, vgg19.preprocess_input),
          (ResNet50, resnet50.preprocess_input)]
results = {}

for Net, prep in models:
    keras.backend.clear_session()
    base = Net(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
    base.trainable = False                       # freeze pretrained features

    model = keras.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    model.fit(prep(X_train.astype('float32')), y_train, epochs=5, batch_size=64, verbose=2)
    acc = model.evaluate(prep(X_test.astype('float32')), y_test, verbose=0)[1]
    results[Net.__name__] = acc
    print(f'{Net.__name__} test accuracy: {acc:.3f}\n')

In [ ]:
for name, acc in results.items():
    print(f'{name:10s} {acc:.3f}')